# 07 — Peaks and their cross-domain specificity (Figure 3)
Five strongest positive local maxima of each unthresholded T map (≥ 20 mm apart), labelled with
Schaefer-1000 (cortex), Tian S4 (subcortex) and Buckner (cerebellum); white-matter peaks get the
nearest grey-matter parcel. Each peak is then evaluated in all six domain maps.

In [ ]:
import sys; sys.path.insert(0, "utils")
from config import *
from figstyle import apply_figure_style, homogenize_fig, pack_rows
import re
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from nilearn import image, plotting
from scipy import stats
from scipy.spatial import cKDTree

brain, affine, BG = load_mask()
mask_img = nib.Nifti1Image(brain.astype(np.uint8), affine)
COLORS = ["#E41A1C", "#377EB8", "#4DAF4A", "#984EA3", "#FF7F00", "#A65628"]
SHK = ["AEF", "PS", "Lang", "VerbM", "VsF", "VsM"]

A = DATA_DIR / "atlases"
def atlas(name):
    return image.resample_to_img(str(A / name), mask_img, interpolation="nearest", force_resample=True,
                                 copy_header=True).get_fdata().astype(int)
sch = atlas("Schaefer2018_1000Parcels_7Networks_order_FSLMNI152_1mm.nii.gz")
tian = atlas("Tian_Subcortex_S4_3T_1mm.nii.gz")
buck = atlas("Buckner7.nii.gz")                                    # used as cerebellum mask only
sch_lab = np.loadtxt(A / "Schaefer2018_1000Parcels_7Networks_order_FSLMNI152_1mm_labels.txt", dtype=str)
tian_lab = np.loadtxt(A / "Tian_Subcortex_S4_3T_1mm_labels.txt", dtype=str)

def direct_label(i):
    if tian[i] > 0: return tian_lab[tian[i] - 1].replace("_", " ")
    if buck[i] > 0: return "Cerebellum"
    if sch[i] > 0: return sch_lab[sch[i] - 1].replace("7Networks_", "").replace("_", " ")

lab_idx = np.argwhere((sch > 0) | (tian > 0) | (buck > 0))
tree = cKDTree(nib.affines.apply_affine(affine, lab_idx))
def label(i):
    lbl = direct_label(i)
    if lbl: return lbl
    near = direct_label(tuple(lab_idx[tree.query(nib.affines.apply_affine(affine, i))[1]]))
    return near if near == "Cerebellum" else f"WM ~ {near}"

def find_peaks(T, n=5, min_sep=20.0):
    """Greedy positive local maxima of T, at least min_sep mm apart."""
    a = np.where(brain & (T > 0), T, -np.inf)
    chosen = []
    for idx in np.array(np.unravel_index(np.argsort(a, axis=None)[::-1], a.shape)).T:
        if a[tuple(idx)] == -np.inf or len(chosen) == n: break
        c = nib.affines.apply_affine(affine, idx)
        if all(np.linalg.norm(c - c0) >= min_sep for c0, _ in chosen):
            chosen.append((c, tuple(int(v) for v in idx)))
    return chosen

In [ ]:
maps = {k: {"T": load_map(LNM, k, F_TSTAT, brain), "fdr": load_map(LNM, k, F_PERM_FDR, brain) >= THR,
            "fwe": load_map(LNM, k, F_PERM_FWE, brain) >= THR} for k in KEYS}
rows = []
for k in KEYS:
    for rank, (c, i) in enumerate(find_peaks(maps[k]["T"]), 1):
        r = {"domain": DOMAINS[k], "domain_key": k, "peak_rank": rank, "MNI_x": c[0], "MNI_y": c[1], "MNI_z": c[2],
             "anatomical_label": label(i)}
        for kk, s in zip(KEYS, SHK):
            r[f"T_{s}"] = round(float(maps[kk]["T"][i]), 2)
            r[f"sigFDR_{s}"], r[f"sigFWE_{s}"] = bool(maps[kk]["fdr"][i]), bool(maps[kk]["fwe"][i])
        r["own_T"] = r[f"T_{SHK[KEYS.index(k)]}"]
        r["other_mean_absT"] = round(np.mean([abs(r[f"T_{s}"]) for kk, s in zip(KEYS, SHK) if kk != k]), 2)
        r["n_other_sigFDR"] = sum(r[f"sigFDR_{s}"] for kk, s in zip(KEYS, SHK) if kk != k)
        rows.append(r)
pk = pd.DataFrame(rows)
pk.to_csv(TAB_DIR / "peaks_crossdomain.csv", index=False)
d = pk.own_T.abs() - pk.other_mean_absT
print(f"own |T| {pk.own_T.abs().mean():.2f} vs other |T| {pk.other_mean_absT.mean():.2f}; "
      f"{(d > 0).sum()}/{len(d)} peaks; Wilcoxon p = {stats.wilcoxon(d).pvalue:.2e}")
pk[["domain", "peak_rank", "MNI_x", "MNI_y", "MNI_z", "own_T", "anatomical_label"]]

Glass-brain panel (a): sagittal views show each hemisphere laterally (left-hemisphere peaks in the right view).

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(18, 15), squeeze=False)
fig.subplots_adjust(hspace=0.92, wspace=0.10, top=0.98, bottom=0.03, left=0.03, right=0.99)
for ax, k, col in zip(axes.ravel(), KEYS, COLORS):
    disp = plotting.plot_glass_brain(None, display_mode="lyrz", axes=ax, radiological=True)
    ax.text(0.0, 1.04, DOMAINS[k], transform=ax.transAxes, va="bottom", fontsize=14)
    sub = pk[pk.domain_key == k]
    for sl in disp.axes.values():
        for _, r in sub.iterrows():
            x, y, z = r.MNI_x, r.MNI_y, r.MNI_z
            if (sl.direction == "l" and x < 0) or (sl.direction == "r" and x > 0): continue
            px, py = {"y": (x, z), "z": (x, y)}.get(sl.direction, (y, z))
            sl.ax.scatter([px], [py], s=160 + 8 * abs(r.own_T), facecolors=col, edgecolors="black", linewidths=0.8, zorder=1000)
            sl.ax.text(px, py, str(r.peak_rank), ha="center", va="center", fontsize=10, color="white", zorder=1001)
    ax.text(0.0, -0.03, "\n".join(f"{r.peak_rank}. {r.anatomical_label}  (T = {r.own_T:+.1f})"
                                  f"{'' if r[f'sigFDR_{SHK[KEYS.index(k)]}'] else '  (n.s.)'}" for _, r in sub.iterrows()),
            transform=ax.transAxes, va="top", fontsize=14, family="monospace")
homogenize_fig(fig); fig.savefig(FIG_DIR / "peaks_glass_brain.png", dpi=300, bbox_inches="tight"); plt.close(fig)

Figure 3: (a) glass brains, (b) peak T in all domain maps (dots: pFDR < 0.05), (c) own vs other domains.

In [ ]:
apply_figure_style()
img = pack_rows(FIG_DIR / "peaks_glass_brain.png", gaps=[40, 40, 110, 40, 40, 110, 40, 40]) / 255.0
LAB = list(SH.values())
FW = 7.2; a_h = FW * 0.955 * img.shape[0] / img.shape[1]; FH = a_h + 3.95
fig = plt.figure(figsize=(FW, FH))
ya0 = 1 - (a_h + 0.25) / FH
ax_a = fig.add_axes([0.035, ya0, 0.955, a_h / FH]); ax_a.imshow(img); ax_a.axis("off")
gb = fig.add_gridspec(1, 2, wspace=0.42, left=0.10, right=0.975, bottom=0.55 / FH, top=ya0 - 0.55 / FH)
ax_b, ax_c = fig.add_subplot(gb[0, 0]), fig.add_subplot(gb[0, 1])
M = pk[[f"T_{s}" for s in SHK]].values; S = pk[[f"sigFDR_{s}" for s in SHK]].values.astype(bool)
VM = 8
im = ax_b.imshow(M, cmap="RdBu_r", vmin=-VM, vmax=VM, aspect="auto", interpolation="nearest")
yy, xx = np.where(S)
rgb = plt.get_cmap("RdBu_r")(Normalize(-VM, VM)(M[yy, xx]))[:, :3]
dark = rgb @ [0.2126, 0.7152, 0.0722] < 0.45                       # white dots on dark tiles
ax_b.scatter(xx, yy, s=3.0, c=np.where(dark[:, None], 1.0, 0.1) * np.ones(3), lw=0, zorder=3)
for i in range(6):
    ax_b.add_patch(plt.Rectangle((i - .5, 5 * i - .5), 1, 5, fill=False, ec="black", lw=1.0, zorder=4))
    if i: ax_b.axhline(5 * i - .5, color="0.6", lw=0.5, zorder=2)
ax_b.set_yticks([5 * i + 2 for i in range(6)]); ax_b.set_yticklabels(LAB)
ax_b.set_xticks(range(6)); ax_b.set_xticklabels(LAB, rotation=45, ha="right"); ax_b.tick_params(length=0)
ax_b.set_ylabel("Top five positive peaks per domain"); ax_b.set_xlabel("Domain map evaluated at the peak voxel")
for s_ in ax_b.spines.values(): s_.set_visible(False)
cb = fig.colorbar(im, ax=ax_b, fraction=0.06, pad=0.03, extend="both", ticks=[-VM, 0, VM])
cb.ax.tick_params(labelsize=6, length=2); cb.set_label("T", fontsize=7, labelpad=1)
own, oth = pk.own_T.abs().values, pk.other_mean_absT.values
for i, k in enumerate(KEYS):
    m = (pk.domain_key == k).values
    ax_c.scatter(oth[m], own[m], s=16, color=COLORS[i], lw=0, zorder=3, label=LAB[i])
ax_c.plot([0, 9], [0, 9], ls="--", color="0.6", lw=0.8, zorder=1)
ax_c.set(xlim=(0, 9), ylim=(0, 9), aspect="equal", xlabel="Mean |T| of the five other domain maps", ylabel="T in own domain map")
ax_c.text(0.97, 0.03, f"{int((own > oth).sum())} of {len(own)} peaks above identity", transform=ax_c.transAxes,
          ha="right", va="bottom", fontsize=6)
ax_c.legend(fontsize=5.5, loc="lower right", bbox_to_anchor=(1.0, 0.09), handletextpad=0.1, labelspacing=0.25)
fig.canvas.draw(); inv = fig.transFigure.inverted(); rr = fig.canvas.get_renderer()
fig.text(0.012, 1 - 0.06 / FH, "a", fontsize=10, va="top")
yt = max(inv.transform(a.get_tightbbox(rr).get_points())[1, 1] for a in (ax_b, ax_c)) + 0.012
for ax, L, ttl in ((ax_b, "b", "Peak T-values across domain maps"), (ax_c, "c", "Own- vs other-domain effect at the peaks")):
    x0 = inv.transform(ax.get_tightbbox(rr).get_points())[0, 0]
    fig.text(0.012 if L == "b" else x0 - 0.01, yt, L, fontsize=10, va="baseline")
    fig.text(inv.transform(ax.get_window_extent(rr).get_points())[0, 0], yt, ttl, fontsize=7.5, va="baseline")
homogenize_fig(fig)
fig.savefig(FIG_DIR / "Fig3_peaks.png", dpi=300); fig.savefig(FIG_DIR / "Fig3_peaks.pdf")